In [ ]:
import pandas as pd
import numpy as np
import ta.momentum
import ta.trend
import ta.volatility
import ta.volume
from datetime import timedelta

# Sample data
data = pd.read_csv('../common/MachineLearningModel/output/five_mins/EURUSD_5_Min_testing.csv')
data['datetime'] = pd.to_datetime(data['datetime'])

# Initialize variables
Gd_228 = 25.0
Gi_220 = 17
Gd_248 = 0.0001 if data['close'].apply(lambda x: len(str(x).split('.')[1]) < 4).all() else 0.00001
EnableAlert = True
SoundFilename = "alert.wav"

# Calculate indicators
data['ATR'] = ta.volatility.AverageTrueRange(data['high'], data['low'], data['close'], window=5).average_true_range()
data['Stochastic'] = ta.momentum.StochasticOscillator(data['high'], data['low'], data['close'], window=20, smooth_window=12).stoch()
data['CCI'] = ta.trend.CCIIndicator(data['high'], data['low'], data['close'], window=80).cci()
data['Momentum_60'] = ta.momentum.ROCIndicator(data['close'], window=60).roc()
data['Momentum_4'] = ta.momentum.ROCIndicator(data['close'], window=4).roc()
data['WPR'] = ta.momentum.WilliamsRIndicator(data['high'], data['low'], data['close'], lbp=14).williams_r()
data['Force'] = ta.volume.ForceIndexIndicator(data['close'], data['volume'], window=13).force_index()
bb = ta.volatility.BollingerBands(data['close'], window=20, window_dev=2)
data['Bollinger_Upper'] = bb.bollinger_hband()
data['Bollinger_Lower'] = bb.bollinger_lband()
data['MA_High'] = ta.trend.EMAIndicator(data['high'], window=1).ema_indicator()
data['MA_Median'] = ta.trend.EMAIndicator((data['high'] + data['low']) / 2, window=1).ema_indicator()
data['MA_Low'] = ta.trend.EMAIndicator(data['low'], window=1).ema_indicator()

# Determine market conditions
def determine_market_condition(stochastic_value):
    if 75.0 >= stochastic_value >= 25.0:
        return "SAFE TRADE"
    elif 88.0 >= stochastic_value > 75.0 or 25.0 > stochastic_value >= 12.0:
        return "S/R AREA"
    elif stochastic_value > 88.0 or stochastic_value < 12.0:
        return "HIGH RISK!"
    else:
        return ""

data['Market_Condition'] = data['Stochastic'].apply(determine_market_condition)

# Buy/Sell Signal Logic
data['Buy_Signal'] = np.nan
data['Sell_Signal'] = np.nan

for i in range(10, len(data)):
    G_high_324 = data['high'][i-10:i].max()
    G_low_332 = data['low'][i-10:i].min()
    Gd_316 = sum((10 - j) * (data['high'][i-j] - data['low'][i-j]) for j in range(10)) / 55.0

    if data['close'][i] > G_high_324 - (G_high_324 - G_low_332) * Gi_220 / 100.0:
        data.at[i, 'Buy_Signal'] = data['high'][i] + Gd_316 / 2.0
    elif data['close'][i] < G_low_332 + (G_high_324 - G_low_332) * Gi_220 / 100.0:
        data.at[i, 'Sell_Signal'] = data['low'][i] - Gd_316 / 2.0


data['UTC'] = pd.to_datetime(data['datetime']) + timedelta(hours=5)
data['GMT'] = data['UTC'] + timedelta(hours=2)
output_file = '../common/MachineLearningModel/output/outputextremearrowEurusd.csv'
data.to_csv(output_file, index=False)


